<!-- codex-architecture-notes -->
## Architectural Notes

**Purpose:** Diagnoses disagreements between final marks and finish/status fields in the student-course table.

**Notebook Shape:** 3 cells (2 code, 1 markdown).

**Inputs / Data Sources:**
- `df = pd.read_parquet(DATA_PATH)`
- `df = pd.read_csv(DATA_PATH, dtype="string")`

**Outputs / Side Effects:**
- `No explicit persisted output detected; side effects are limited to notebook display state unless cells are edited.`

**Logic Flow:**
1. Load raw or cleaned student-course data.
2. Compare mark values against finish/status indicators.
3. Inspect disagreement examples.

**Maintainability Notes:** Findings should feed explicit cleaning rules; otherwise target labels can remain inconsistent.


# Mark vs Finish Status Disagreement Diagnostic

Audit-only diagnostic for disagreement between `final_mark >= 50` and official `finish_status == "P"`.

This notebook does not write data and does not add mismatch columns as model features. Current target design remains:

- M1 target = `final_mark >= 50`
- M2 target = `final_mark`
- `finish_status` is not used as the main target.

In [2]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

# Default raw V_CRG_STUDENT_COURSE source used by the preprocessing notebooks.
# If the audit target is an enriched or pre-enriched training file, point DATA_PATH there instead.
DATA_PATH = Path(r"D:\AI\Real projects\Academic_Advisor\data\features\merged_add_acd_crg.parquet")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"DATA_PATH does not exist: {DATA_PATH}")

suffix = DATA_PATH.suffix.lower()
if suffix == ".parquet":
    df = pd.read_parquet(DATA_PATH)
elif suffix == ".csv":
    df = pd.read_csv(DATA_PATH, dtype="string")
else:
    raise ValueError("DATA_PATH must point to a .parquet or .csv file.")

df = df.copy()
df.columns = df.columns.str.lower()

required_columns = {"final_mark", "finish_status"}
missing_required = sorted(required_columns - set(df.columns))
if missing_required:
    raise KeyError(f"Missing required columns: {missing_required}")

final_mark = pd.to_numeric(df["final_mark"], errors="coerce")
finish_status = df["finish_status"].astype("string").str.strip().str.upper().replace("", pd.NA)

analysis_mask = final_mark.notna() & finish_status.notna()
mark_based_pass = final_mark.ge(50)
official_pass = finish_status.eq("P")
mismatch_mask = analysis_mask & mark_based_pass.ne(official_pass)

total_rows = int(analysis_mask.sum())
mismatch_count = int(mismatch_mask.sum())
mismatch_pct = (mismatch_count / total_rows * 100) if total_rows else 0.0

mark_ge_50_not_p = int((analysis_mask & mark_based_pass & ~official_pass).sum())
mark_lt_50_p = int((analysis_mask & ~mark_based_pass & official_pass).sum())

print(f"DATA_PATH: {DATA_PATH}")
print(f"Total rows with non-null final_mark and finish_status: {total_rows:,}")
print(f"Mismatch count: {mismatch_count:,}")
print(f"Mismatch percentage: {mismatch_pct:.4f}%")
print(f"final_mark >= 50 but finish_status != 'P': {mark_ge_50_not_p:,}")
print(f"final_mark < 50 but finish_status == 'P': {mark_lt_50_p:,}")

print("\nMismatch breakdown by finish_status:")
finish_status_breakdown = (
    finish_status[mismatch_mask]
    .value_counts(dropna=False)
    .rename_axis("finish_status")
    .reset_index(name="mismatch_count")
)
finish_status_breakdown["mismatch_pct"] = (
    finish_status_breakdown["mismatch_count"].div(mismatch_count).mul(100) if mismatch_count else 0.0
)
display(finish_status_breakdown)

def mismatch_concentration(column: str, top_n: int = 20) -> pd.DataFrame:
    mismatch_counts = df.loc[mismatch_mask, column].value_counts(dropna=False).rename("mismatch_count")
    total_counts = df.loc[analysis_mask, column].value_counts(dropna=False).rename("total_rows")
    concentration = (
        pd.concat([mismatch_counts, total_counts], axis=1)
        .fillna(0)
        .astype({"mismatch_count": "int64", "total_rows": "int64"})
        .reset_index()
        .rename(columns={"index": column})
    )
    concentration["mismatch_pct_of_all_mismatches"] = (
        concentration["mismatch_count"].div(mismatch_count).mul(100) if mismatch_count else 0.0
    )
    concentration["mismatch_pct_within_group"] = (
        concentration["mismatch_count"].div(concentration["total_rows"].replace(0, pd.NA)).mul(100)
    )
    return concentration.sort_values(
        ["mismatch_count", "mismatch_pct_within_group"],
        ascending=[False, False],
    ).head(top_n)

for column in ["degree_id", "course_id", "part_year", "part_id", "faculty_id"]:
    print(f"\nTop mismatch concentrations by {column}:")
    if column not in df.columns:
        print(f"Column not found: {column}")
        continue
    display(mismatch_concentration(column))

DATA_PATH: D:\AI\Real projects\Academic_Advisor\data\features\merged_add_acd_crg.parquet
Total rows with non-null final_mark and finish_status: 761,347
Mismatch count: 1,780
Mismatch percentage: 0.2338%
final_mark >= 50 but finish_status != 'P': 1,780
final_mark < 50 but finish_status == 'P': 0

Mismatch breakdown by finish_status:


,finish_status,mismatch_count,mismatch_pct
0,F,1185,66.573034
1,FE,595,33.426966



Top mismatch concentrations by degree_id:


,degree_id,mismatch_count,total_rows,mismatch_pct_of_all_mismatches,mismatch_pct_within_group
0,18.111,353,2567,19.831461,13.751461
1,6.111,352,18192,19.775281,1.934916
2,21.111,183,5920,10.280899,3.091216
3,3.111,182,39547,10.224719,0.460212
4,2.111,177,134214,9.943820,0.131879
5,1.111,73,179865,4.101124,0.040586
6,8.111,55,21125,3.089888,0.260355
7,11.111,54,15920,3.033708,0.339196
8,22.111,33,6831,1.853933,0.483092
9,26.111,31,9665,1.741573,0.320745



Top mismatch concentrations by course_id:


,course_id,mismatch_count,total_rows,mismatch_pct_of_all_mismatches,mismatch_pct_within_group
0,1093.111,141,1340,7.921348,10.522388
1,385.111,24,148,1.348315,16.216216
2,384.111,23,168,1.292135,13.690476
4,1180.111,22,1220,1.235955,1.803279
3,432.111,22,1962,1.235955,1.121305
6,377.111,21,108,1.179775,19.444444
5,516.111,21,1281,1.179775,1.639344
7,121.111,20,1190,1.123596,1.680672
9,424.111,19,151,1.067416,12.582781
10,368.111,19,152,1.067416,12.500000



Top mismatch concentrations by part_year:
Column not found: part_year

Top mismatch concentrations by part_id:


,part_id,mismatch_count,total_rows,mismatch_pct_of_all_mismatches,mismatch_pct_within_group
0,20111,122,6680,6.853933,1.826347
1,20241,112,37293,6.292135,0.300324
2,20102,108,1071,6.067416,10.084034
3,20101,94,1244,5.280899,7.556270
4,20231,80,37348,4.494382,0.214202
5,20112,76,7252,4.269663,1.047987
6,20221,68,35506,3.820225,0.191517
7,20222,63,34222,3.539326,0.184092
8,20081,58,504,3.258427,11.507937
9,20092,58,1116,3.258427,5.197133



Top mismatch concentrations by faculty_id:


,faculty_id,mismatch_count,total_rows,mismatch_pct_of_all_mismatches,mismatch_pct_within_group
0,5.111,1101,69291,61.853933,1.588951
1,7.111,216,66516,12.134831,0.324734
2,2.111,180,154629,10.112360,0.116408
3,167.111,155,37216,8.707865,0.416488
4,3.111,74,205630,4.157303,0.035987
5,177.111,34,13980,1.910112,0.243205
6,6.111,11,43567,0.617978,0.025248
7,4.111,7,167458,0.393258,0.004180
8,187.111,2,2520,0.112360,0.079365
9,195.111,0,540,0.000000,0.000000
